# AI-Powered Customer Churn Intelligence System

## Day 7 - Reusable AI Churn Intelligence Pipeline

This notebook converts the individual Customer 2461 prototype into a
reusable customer-level intelligence pipeline.

The pipeline performs:

1. Customer churn prediction
2. Risk classification
3. SHAP-based explanation
4. LLM-powered retention recommendation

The goal is to allow the same workflow to be applied to any customer
in the dataset.

In [1]:
import os
import json
import requests
import joblib
import shap
import pandas as pd
import numpy as np

from pathlib import Path
from dotenv import load_dotenv

from sklearn.model_selection import train_test_split

In [2]:
print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
load_dotenv(override=True)

api_key = os.getenv("OPENROUTER_API_KEY")
model_name = os.getenv("OPENROUTER_MODEL", "openrouter/free")

if not api_key:
    raise ValueError(
        "OPENROUTER_API_KEY was not found."
    )

print("OpenRouter configuration loaded.")
print("Model:", model_name)

OpenRouter configuration loaded.
Model: openrouter/free


In [4]:
df = pd.read_csv(
    "../data/customer_churn_processed.csv"
)

logistic_model = joblib.load(
    "../models/logistic_regression_churn_model.pkl"
)

print("Dataset shape:", df.shape)
print("Model loaded successfully.")

Dataset shape: (2800, 15)
Model loaded successfully.


In [5]:
X = df.drop(
    columns=[
        "user_id",
        "churn",
        "churn_target"
    ]
)

y = df["churn_target"]

X["signup_date"] = pd.to_datetime(
    X["signup_date"],
    errors="coerce"
)

X["signup_year"] = X["signup_date"].dt.year

X = X.drop(
    columns=["signup_date"]
)

print("Feature shape:", X.shape)

Feature shape: (2800, 12)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Test customers:", len(X_test))

Test customers: 560


In [7]:
preprocessor = logistic_model.named_steps[
    "preprocessor"
]

classifier = logistic_model.named_steps[
    "classifier"
]

X_test_transformed = preprocessor.transform(
    X_test
)

feature_names = (
    preprocessor
    .get_feature_names_out()
)

X_test_transformed_df = pd.DataFrame(
    X_test_transformed.toarray()
    if hasattr(X_test_transformed, "toarray")
    else X_test_transformed,
    columns=feature_names,
    index=X_test.index
)

explainer = shap.LinearExplainer(
    classifier,
    X_test_transformed_df
)

shap_values = explainer(
    X_test_transformed_df
)

print("SHAP explainer ready.")

SHAP explainer ready.


## 1. Customer Risk Classification

Customers are classified according to predicted churn probability.

- High Risk: >= 70%
- Medium Risk: 40% to <70%
- Low Risk: <40%

These thresholds are business segmentation rules.

In [8]:
def classify_risk(probability):
    if probability >= 0.70:
        return "High Risk"
    elif probability >= 0.40:
        return "Medium Risk"
    else:
        return "Low Risk"

In [9]:
def predict_customer_risk(customer_id):
    
    if customer_id not in X_test.index:
        raise ValueError(
            f"Customer {customer_id} is not in the test set."
        )
    
    position = X_test.index.get_loc(customer_id)
    
    probability = logistic_model.predict_proba(
        X_test.loc[[customer_id]]
    )[0, 1]
    
    prediction = logistic_model.predict(
        X_test.loc[[customer_id]]
    )[0]
    
    risk = classify_risk(probability)
    
    return {
        "customer_id": int(customer_id),
        "predicted_churn": int(prediction),
        "churn_probability": round(
            float(probability),
            4
        ),
        "risk_category": risk,
        "shap_position": int(position)
    }

In [12]:
def predict_customer_risk(customer_id):

    customer_row = df[
        df["user_id"] == customer_id
    ]

    if customer_row.empty:
        raise ValueError(
            f"Customer {customer_id} was not found."
        )

    customer_features = customer_row.drop(
        columns=[
            "user_id",
            "churn",
            "churn_target"
        ]
    ).copy()

    customer_features["signup_date"] = pd.to_datetime(
        customer_features["signup_date"],
        errors="coerce"
    )

    customer_features["signup_year"] = (
        customer_features["signup_date"].dt.year
    )

    customer_features = customer_features.drop(
        columns=["signup_date"]
    )

    probability = logistic_model.predict_proba(
        customer_features
    )[0, 1]

    prediction = logistic_model.predict(
        customer_features
    )[0]

    risk = classify_risk(probability)

    return {
        "customer_id": int(customer_id),
        "predicted_churn": int(prediction),
        "churn_probability": round(
            float(probability),
            4
        ),
        "risk_category": risk
    }

In [13]:
customer_prediction = predict_customer_risk(2461)

customer_prediction

{'customer_id': 2461,
 'predicted_churn': 1,
 'churn_probability': 0.9587,
 'risk_category': 'High Risk'}

In [14]:
def explain_customer(customer_id, top_n=5):

    customer_row = df[
        df["user_id"] == customer_id
    ]

    if customer_row.empty:
        raise ValueError(
            f"Customer {customer_id} was not found."
        )

    customer_features = customer_row.drop(
        columns=[
            "user_id",
            "churn",
            "churn_target"
        ]
    ).copy()

    customer_features["signup_date"] = pd.to_datetime(
        customer_features["signup_date"],
        errors="coerce"
    )

    customer_features["signup_year"] = (
        customer_features["signup_date"].dt.year
    )

    customer_features = customer_features.drop(
        columns=["signup_date"]
    )

    # Transform customer using the same preprocessing
    customer_transformed = preprocessor.transform(
        customer_features
    )

    customer_transformed_df = pd.DataFrame(
        customer_transformed.toarray()
        if hasattr(customer_transformed, "toarray")
        else customer_transformed,
        columns=feature_names
    )

    # Calculate SHAP explanation
    customer_shap = explainer(
        customer_transformed_df
    )

    shap_values_customer = customer_shap.values[0]

    explanation = pd.DataFrame({
        "feature": feature_names,
        "shap_value": shap_values_customer
    })

    explanation["absolute_shap"] = (
        explanation["shap_value"].abs()
    )

    explanation = (
        explanation
        .sort_values(
            "absolute_shap",
            ascending=False
        )
        .reset_index(drop=True)
    )

    return explanation.head(top_n)

In [15]:
customer_shap = explain_customer(
    2461,
    top_n=5
)

customer_shap

,feature,shap_value,absolute_shap
0,cat__usage_level_Low,1.268278,1.268278
1,num__support_tickets,0.966247,0.966247
2,num__payment_failures,0.587648,0.587648
3,cat__login_recency_category_Inactive,0.324414,0.324414
4,cat__support_risk_Low,-0.253696,0.253696


In [16]:
second_customer_id = int(
    df["user_id"].iloc[0]
)

print("Testing customer:", second_customer_id)

print(
    predict_customer_risk(
        second_customer_id
    )
)

print(
    explain_customer(
        second_customer_id,
        top_n=5
    )
)

Testing customer: 1
{'customer_id': 1, 'predicted_churn': 1, 'churn_probability': 0.5674, 'risk_category': 'Medium Risk'}
                                feature  shap_value  absolute_shap
0                  cat__usage_level_Low    1.268278       1.268278
1                 num__payment_failures   -0.334153       0.334153
2           num__avg_weekly_usage_hours   -0.285477       0.285477
3  cat__login_recency_category_Inactive   -0.254897       0.254897
4                 cat__support_risk_Low   -0.253696       0.253696


In [18]:
def build_customer_context(customer_id):

    customer = df[
        df["user_id"] == customer_id
    ]

    if customer.empty:
        raise ValueError(
            f"Customer {customer_id} was not found."
        )

    customer = customer.iloc[0]

    prediction = predict_customer_risk(
        customer_id
    )

    explanation = explain_customer(
        customer_id,
        top_n=5
    )

    context = {
        "customer_id": int(
            customer["user_id"]
        ),
        "plan_type": customer["plan_type"],
        "monthly_fee": float(
            customer["monthly_fee"]
        ),
        "avg_weekly_usage_hours": float(
            customer["avg_weekly_usage_hours"]
        ),
        "support_tickets": int(
            customer["support_tickets"]
        ),
        "payment_failures": int(
            customer["payment_failures"]
        ),
        "tenure_months": int(
            customer["tenure_months"]
        ),
        "last_login_days_ago": int(
            customer["last_login_days_ago"]
        ),
        "login_recency_category": customer[
            "login_recency_category"
        ],
        "support_risk": customer[
            "support_risk"
        ],
        "payment_risk": customer[
            "payment_risk"
        ],
        "usage_level": customer[
            "usage_level"
        ],
        "churn_probability": prediction[
            "churn_probability"
        ],
        "risk_category": prediction[
            "risk_category"
        ],
        "top_shap_drivers": [
            {
                "feature": row["feature"],
                "shap_value": round(
                    float(row["shap_value"]),
                    4
                )
            }
            for _, row in explanation.iterrows()
        ]
    }

    return context

In [19]:
context_customer_1 = build_customer_context(1)

print(
    json.dumps(
        context_customer_1,
        indent=4
    )
)

{
    "customer_id": 1,
    "plan_type": "Premium",
    "monthly_fee": 699.0,
    "avg_weekly_usage_hours": 1.1,
    "support_tickets": 4,
    "payment_failures": 1,
    "tenure_months": 8,
    "last_login_days_ago": 14,
    "login_recency_category": "Active",
    "support_risk": "Medium",
    "payment_risk": "Medium",
    "usage_level": "Low",
    "churn_probability": 0.5674,
    "risk_category": "Medium Risk",
    "top_shap_drivers": [
        {
            "feature": "cat__usage_level_Low",
            "shap_value": 1.2683
        },
        {
            "feature": "num__payment_failures",
            "shap_value": -0.3342
        },
        {
            "feature": "num__avg_weekly_usage_hours",
            "shap_value": -0.2855
        },
        {
            "feature": "cat__login_recency_category_Inactive",
            "shap_value": -0.2549
        },
        {
            "feature": "cat__support_risk_Low",
            "shap_value": -0.2537
        }
    ]
}


In [21]:
def generate_retention_recommendation(customer_context):

    system_prompt = """
You are a customer retention intelligence assistant.

Analyze the supplied customer information and generate a practical,
personalized retention strategy.

Use ONLY the information provided.

Rules:
- Do not invent customer information.
- Do not assume support tickets are open or unresolved.
- Do not claim SHAP values prove causation.
- Do not describe a customer as certain or guaranteed to churn.
- Describe churn as a predicted likelihood.
- Recommendations must directly address the strongest SHAP drivers.
- Keep recommendations practical and concise.

Return exactly these sections:

1. CHURN ASSESSMENT
2. KEY CHURN REASONS
3. RETENTION ACTIONS
4. PRIORITY
5. BUSINESS EXPLANATION
"""

    user_prompt = f"""
Analyze this customer:

{json.dumps(customer_context, indent=4)}

Generate a personalized retention recommendation based only on
the supplied customer information and SHAP drivers.
"""

    payload = {
        "model": model_name,
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "temperature": 0.2
    }

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },
        json=payload,
        timeout=60
    )

    print("Status code:", response.status_code)

    response.raise_for_status()

    result = response.json()

    return result["choices"][0]["message"]["content"]

In [22]:
recommendation_customer_1 = (
    generate_retention_recommendation(
        context_customer_1
    )
)

print(recommendation_customer_1)

Status code: 200
**1. CHURN ASSESSMENT**  
- Churn probability: **56.7%** (medium risk).  
- Risk category: **Medium Risk**.

**2. KEY CHURN REASONS**  
- **Low usage** (cat__usage_level_Low) – the strongest SHAP driver, indicating limited product engagement.  
- **One payment failure** (num__payment_failures) – signals payment friction.  
- **Medium support risk** (4 support tickets) – potential unresolved issues.  
- **Recency of login** (currently active) – if activity drops, churn risk would rise.

**3. RETENTION ACTIONS**  
1. **Usage‑focused outreach** – schedule a quick usage review (e.g., webinar or personalized tip session) to help the customer discover higher‑value features and increase weekly usage hours.  
2. **Payment verification** – contact the customer to confirm the payment method and resolve the single failure, eliminating payment‑related friction.  
3. **Support follow‑up** – check the status of the four support tickets, ensure any open issues are closed, and confirm

In [23]:
def analyze_customer(customer_id):
    
    # 1. Build customer context
    context = build_customer_context(
        customer_id
    )
    
    # 2. Generate ML prediction
    prediction = predict_customer_risk(
        customer_id
    )
    
    # 3. Generate SHAP explanation
    shap_explanation = explain_customer(
        customer_id,
        top_n=5
    )
    
    return {
        "customer_id": customer_id,
        "prediction": prediction,
        "customer_context": context,
        "shap_explanation": shap_explanation
    }

In [24]:
customer_analysis = analyze_customer(1)

print(
    json.dumps(
        customer_analysis["prediction"],
        indent=4
    )
)

print("\nTop SHAP Drivers:")
print(
    customer_analysis["shap_explanation"]
)

{
    "customer_id": 1,
    "predicted_churn": 1,
    "churn_probability": 0.5674,
    "risk_category": "Medium Risk"
}

Top SHAP Drivers:
                                feature  shap_value  absolute_shap
0                  cat__usage_level_Low    1.268278       1.268278
1                 num__payment_failures   -0.334153       0.334153
2           num__avg_weekly_usage_hours   -0.285477       0.285477
3  cat__login_recency_category_Inactive   -0.254897       0.254897
4                 cat__support_risk_Low   -0.253696       0.253696


In [27]:
def generate_customer_report(customer_id):
    
    analysis = analyze_customer(customer_id)

    context = analysis["customer_context"]

    recommendation = generate_retention_recommendation(
        context
    )

    report = {
        "customer_id": customer_id,
        "prediction": analysis["prediction"],
        "top_shap_drivers": (
            analysis["shap_explanation"]
            .to_dict(orient="records")
        ),
        "llm_recommendation": recommendation
    }

    return report